# [STARTER] Udaplay Project

## Part 02 - Agent

In this part of the project, you'll use your VectorDB to be part of your Agent as a tool.

You're building UdaPlay, an AI Research Agent for the video game industry. The agent will:
1. Answer questions using internal knowledge (RAG)
2. Search the web when needed
3. Maintain conversation state
4. Return structured outputs
5. Store useful information for future use

### Setup

In [1]:
# Only needed for Udacity workspace

import importlib.util
import sys

# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [9]:


import os
import json
from dotenv import load_dotenv
from openai import OpenAI
from tavily import TavilyClient

from lib.agents import Agent
from lib.llm import LLM
from lib.messages import UserMessage, SystemMessage, ToolMessage, AIMessage
from lib.tooling import tool

In [10]:

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")

### Tools

Build at least 3 tools:
- retrieve_game: To search the vector DB
- evaluate_retrieval: To assess the retrieval performance
- game_web_search: If no good, search the web


#### Retrieve Game Tool

In [11]:
chroma_client = chromadb.PersistentClient(path="chromadb")
collection = chroma_client.get_collection("udaplay_games")

openai_client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"), base_url="https://openai.vocareum.com/v1")

def embed(texts):
    resp = openai_client.embeddings.create(model="text-embedding-3-small", input=texts)
    return [d.embedding for d in resp.data]


@tool
def retrieve_game(query: str) -> list:
    """
    Semantic search: Finds most results in the vector DB
    args:
    - query: a question about game industry.

    You'll receive results as list. Each element contains:
    - Platform: like Game Boy, Playstation 5, Xbox 360...)
    - Name: Name of the Game
    - YearOfRelease: Year when that game was released for that platform
    - Description: Additional details about the game
    """
    results = collection.query(
        query_embeddings=embed([query]),
        n_results=5,
        include=["metadatas", "distances"],
    )
    games = []
    for meta, dist in zip(results["metadatas"][0], results["distances"][0]):
        games.append({
            "Platform": meta.get("Platform"),
            "Name": meta.get("Name"),
            "YearOfRelease": meta.get("YearOfRelease"),
            "Description": meta.get("Description"),
            "Genre": meta.get("Genre"),
            "Publisher": meta.get("Publisher"),
            "similarity": round(1 - dist, 3),
        })
    return games

#### Evaluate Retrieval Tool

In [12]:
from pydantic import BaseModel

class EvaluationReport(BaseModel):
    useful: bool
    description: str


@tool
def evaluate_retrieval(question: str, retrieved_docs: list) -> dict:
    """
    Based on the user's question and on the list of retrieved documents,
    it will analyze the usability of the documents to respond to that question.
    args:
    - question: original question from user
    - retrieved_docs: retrieved documents most similar to the user query in the Vector Database
    The result includes:
    - useful: whether the documents are useful to answer the question
    - description: description about the evaluation result
    """
    # Quick check first: if nothing is even close, don't spend an LLM call
    best = max((d.get("similarity", 1) for d in retrieved_docs if isinstance(d, dict)), default=0)
    if not retrieved_docs or best < 0.25:
        return EvaluationReport(
            useful=False,
            description="No retrieved document is close to the question; search the web instead.",
        ).model_dump()

    prompt = (
        "Your task is to evaluate if the documents are enough to respond the query. "
        "Give a detailed explanation, so it's possible to take an action to accept it or not.\n"
        "Judge only from the documents, not from your own knowledge. If a needed detail "
        "(e.g. the specific game, platform, date or publisher asked about) is missing, useful must be false.\n\n"
        'Respond with JSON only: {"useful": true/false, "description": "..."}\n\n'
        f"Question: {question}\n\n"
        f"Documents:\n{json.dumps(retrieved_docs, ensure_ascii=False, indent=2)}"
    )

    response = openai_client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        response_format={"type": "json_object"},
        temperature=0,
    )
    report = EvaluationReport.model_validate_json(response.choices[0].message.content)
    return report.model_dump()

#### Game Web Search Tool

In [13]:
tavily_client = TavilyClient(api_key=os.getenv("TAVILY_API_KEY"))

@tool
def game_web_search(question: str) -> dict:
    """
    Web search: searches the internet for information about the game industry.
    Use it when the local vector DB doesn't have enough information to answer.
    args:
    - question: a question about game industry.

    Returns a short answer summary plus a list of web results, each with
    title, url, content and score. Cite the url of any result you use.
    """
    try:
        response = tavily_client.search(
            query=question,
            search_depth="advanced",
            max_results=5,
            include_answer=True,
        )
    except Exception as e:
        return {"error": f"Web search failed: {e}", "results": []}

    return {
        "answer": response.get("answer"),
        "results": [
            {
                "title": r.get("title"),
                "url": r.get("url"),
                "content": r.get("content"),
                "score": r.get("score"),
            }
            for r in response.get("results", [])
        ],
    }

### Agent

In [ ]:
# TODO: Create your Agent abstraction using StateMachine
# Equip with an appropriate model
# Craft a good set of instructions 
# Plug all Tools you developed

In [14]:
instructions = """
You are UdaPlay, an AI research agent that answers questions about video games:
titles, release dates, platforms, descriptions, genres and publishers.

Always follow this process:
1. Call retrieve_game with the user's question to search the local game database.
For follow-up questions ("who published it?"), rewrite the query to name the game.
2. Call evaluate_retrieval with the question and the documents retrieve_game returned.
3. If evaluate_retrieval says useful is false, call game_web_search with the question.
Never search the web before checking the local database.
4. Answer using ONLY the information returned by the tools. Do not use your own knowledge.

How to write the answer:
- Start with a direct answer, then supporting details, in natural, readable prose.
- Cite every source:
- (Source: local game database - <game name>) for retrieve_game results
- (Source: <page title> - <url>) for web results
- If you combined local and web information, cite both.
- End with a confidence line:
- "Confidence: High" if useful local documents or several agreeing web sources answered it
- "Confidence: Medium" if it relied on a single web source or sources partly disagree
-"Confidence: Low" if nothing fully answered the question, and say what is missing
"""

agent = Agent(
    model_name="gpt-4o-mini",
    instructions=instructions,
    tools=[retrieve_game, evaluate_retrieval, game_web_search],
)

### (Optional) Advanced

In [17]:
# helpers and long-term memory
import datetime

def call(t, **kwargs):
    """Call a tool whether or not @tool wrapped it."""
    fn = getattr(t, "func", None) or getattr(t, "fn", None) or t
    return fn(**kwargs)

# Long-term memory: facts learned from the web, saved between sessions
memory = chroma_client.get_or_create_collection(
    name="udaplay_memory", embedding_function=None, metadata={"hnsw:space": "cosine"}
)

def remember(question, answer, sources):
    memory.upsert(
        ids=[f"mem-{datetime.datetime.now().timestamp()}"],
        documents=[f"Q: {question}\nA: {answer}"],
        embeddings=embed([question]),
        metadatas=[{"question": question, "sources": json.dumps(sources),
                    "saved_at": datetime.datetime.now().isoformat()}],
    )

def recall(question, min_similarity=0.85):
    if memory.count() == 0:
        return []
    r = memory.query(query_embeddings=embed([question]), n_results=2,
                include=["documents", "metadatas", "distances"])
    return [
        {"memory": doc, "sources": json.loads(meta["sources"]), "similarity": round(1 - dist, 3)}
        for doc, meta, dist in zip(r["documents"][0], r["metadatas"][0], r["distances"][0])
        if 1 - dist >= min_similarity
    ]

def llm(prompt, json_mode=False):
    kwargs = {"response_format": {"type": "json_object"}} if json_mode else {}
    r = openai_client.chat.completions.create(
        model="gpt-4o-mini", temperature=0,
        messages=[{"role": "user", "content": prompt}], **kwargs)
    return r.choices[0].message.content

In [18]:
#the state machine
class UdaPlayStateMachine:
    """
    START -> contextualize -> recall_memory -> retrieve -> evaluate
        -> (useful? answer : web_search -> answer) -> save_memory -> END
    """

    def __init__(self):
        self.history = []  # short-term memory: this conversation

    # ----- nodes -------------------------------------------------------
    def contextualize(self, s):
        """Rewrite follow-ups ('who published it?') into standalone questions."""
        if self.history:
            convo = "\n".join(f"{m['role']}: {m['content']}" for m in self.history[-6:])
            s["query"] = llm(f"Conversation:\n{convo}\n\nRewrite this follow-up as a standalone "
                        f"question about a video game. Reply with the question only.\n\n"
                        f"Follow-up: {s['question']}").strip()
        else:
            s["query"] = s["question"]
        return "recall_memory"

    def recall_memory(self, s):
        s["memories"] = recall(s["query"])
        return "retrieve"

    def retrieve(self, s):
        s["docs"] = call(retrieve_game, query=s["query"])
        return "evaluate"

    def evaluate(self, s):
        docs = s["docs"] + s["memories"]
        s["evaluation"] = call(evaluate_retrieval, question=s["query"], retrieved_docs=docs)
        return "answer" if s["evaluation"]["useful"] else "web_search"

    def web_search(self, s):
        s["web"] = call(game_web_search, question=s["query"])
        return "answer"

    def answer(self, s):
        evidence = {
            "local_game_database": s["docs"] if s["evaluation"]["useful"] else [],
            "long_term_memory": s["memories"],
            "web_results": s.get("web", {}),
            "evaluation": s["evaluation"],
        }
        s["answer"] = llm(
            f"{instructions}\n\nQuestion: {s['query']}\n\n"
            f"Evidence gathered by the tools:\n{json.dumps(evidence, ensure_ascii=False, indent=2)}\n\n"
            "Write the final answer now, following the rules above."
        )
        return "save_memory"

    def save_memory(self, s):
        if s.get("web", {}).get("results"):   # only remember things learned from the web
            urls = [r["url"] for r in s["web"]["results"][:3]]
            remember(s["query"], s["answer"], urls)
        return None  # END

    # ----- runner ------------------------------------------------------
    def invoke(self, question, verbose=True):
        state, node = {"question": question}, "contextualize"
        while node:
            if verbose:
                print(f"-> {node}")
            node = getattr(self, node)(state)
        self.history += [{"role": "user", "content": question},
                        {"role": "assistant", "content": state["answer"]}]
        return state

    def reset(self):
        self.history = []


udaplay = UdaPlayStateMachine()

In [19]:
for q in ["When was Pokémon Gold and Silver released?",
        "Which one was the first 3D platformer Mario game?",
        "Was Mortal Kombat X released for PlayStation 5?"]:
    print(f"\nQ: {q}")
    print(udaplay.invoke(q)["answer"])
    print("-" * 60)

# Short-term memory: follow-up questions
print(udaplay.invoke("Who published it?")["answer"])

# Long-term memory: ask the web question again, this time it's answered from memory
udaplay.reset()
print(udaplay.invoke("Was Mortal Kombat X released for PlayStation 5?")["answer"])


Q: When was Pokémon Gold and Silver released?
-> contextualize
-> recall_memory
-> retrieve
-> evaluate
-> answer
-> save_memory
Pokémon Gold and Silver were released in 1999 for the Game Boy Color. These games are part of the second generation of Pokémon, introducing new regions, Pokémon, and gameplay mechanics (Source: local game database - Pokémon Gold and Silver).

Confidence: High
------------------------------------------------------------

Q: Which one was the first 3D platformer Mario game?
-> contextualize
-> recall_memory
-> retrieve
-> evaluate
-> answer
-> save_memory
The first 3D platformer Mario game is "Super Mario 64," which was released in 1996 for the Nintendo 64. This game is notable for being a groundbreaking title in the genre, setting new standards for 3D platforming. In "Super Mario 64," players control Mario as he embarks on a quest to rescue Princess Peach, navigating through various 3D environments and completing challenges (Source: local game database - Supe